# AGENTIC AI
## SESSION 14

Assignment questions and answers


## Question 1

Install Python's 'mcp' package (or the official MCP reference implementation), then start a basic MCP Server locally that responds with a static message when any request is received.

**Answer:**

1. **Installation:**
```bash
pip install mcp
```

2. **Server Code (`server.py`):**
```python
from mcp.server.fastmcp import FastMCP

# Initialize the MCP Server
mcp = FastMCP("StaticResponseServer")

# Tool that returns a static message regardless of input
@mcp.tool()
def get_message() -> str:
    """Responds with a static message."""
    return "This is a static response from the local MCP server."

if __name__ == "__main__":
    mcp.run()
```

3. **Run the Server:**
```bash
python server.py
```

## Question 2

Create a simple MCP Client script that sends a JSON-RPC request to your MCP Server using HTTP transport and prints the response in the console.

Hint: Use the requests library in Python to send the JSON payload.

**Answer:**

```python
import requests

url = "http://localhost:8000/mcp"

payload = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
    "params": {}
}

response = requests.post(url, json=payload)
print(response.json())
```

## Question 3

Modify your MCP Server to recognize a 'get_song_recommendation' method: when called, it should return a JSON object with a random trending song title (imagine a Spotify-style feature).

**Answer:**

```python
import random
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("MusicServer")

TRENDING_SONGS = [
    "Espresso - Sabrina Carpenter",
    "Birds of a Feather - Billie Eilish",
    "Good Luck, Babe! - Chappell Roan",
    "Not Like Us - Kendrick Lamar",
    "A Bar Song (Tipsy) - Shaboozey",
]


@mcp.tool()
def get_song_recommendation() -> dict:
    """Returns a random trending song recommendation."""
    song = random.choice(TRENDING_SONGS)
    return {"trending_song": song}


if __name__ == "__main__":
    mcp.run()
```

## Question 4

Compare stdio transport and HTTP transport in MCP by writing a short note on when you would use each, and update your MCP Client to use stdio transport instead of HTTP for sending a request.

**Answer:**

### When to Use Each Transport

| Transport | When to Use |
| :--- | :--- |
| **stdio** | Use for local tools running on the same machine as the client (e.g., CLI tools, local scripts) where simple process management and low overhead are needed without network exposure. |
| **HTTP (SSE)** | Use for remote tools, cloud services, multi-user architectures, or distributed environments where services run on different machines and require network security. |

---

### MCP Client Using stdio Transport

```python
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


async def main():
    # Configure the local server command
    server_params = StdioServerParameters(
        command="python",  # Executable
        args=["server.py"],  # Server script path
        env=None,
    )

    # Connect via stdio transport instead of HTTP
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Send a request (call a tool)
            result = await session.call_tool(
                name="example_tool", arguments={"param": "value"}
            )
            print(result)


asyncio.run(main())
```

## Question 5

Explore the official MCP documentation or use ChatGPT to draft a custom MCP Prompt for a 'get_movie_showtimes' method (BookMyShow-style). Share the prompt and explain how you would integrate it into your MCP Server.

**Answer:**

### Custom MCP Prompt Definition

```json
{
  "name": "get_movie_showtimes",
  "description": "Finds current movie showtimes, cinema locations, and ticket availability for a specified city and date.",
  "arguments": [
    {"name": "movie_title", "description": "The title of the movie", "required": true},
    {"name": "city", "description": "The city to search for theaters", "required": true},
    {"name": "date", "description": "The date for the showtime in YYYY-MM-DD format", "required": false},
    {"name": "preferred_format", "description": "Format such as 2D, 3D, IMAX, or 4DX", "required": false}
  ]
}
```

### Integration into an MCP Server

* **Register the Prompt:** Add the prompt name, description, and argument schema to the server's prompt registry so it appears in the `prompts/list` MCP capability response.
* **Implement the `prompts/get` Handler:** Write a server handler that listens for requests to fetch `get_movie_showtimes`, validates the input arguments (like `movie_title` and `city`), and returns the formatted user/assistant prompt template.
* **Link with Tools:** Configure the resulting prompt template to instruct the AI agent to call your server's underlying BookMyShow-style API tools to fetch real-time theater data based on the provided arguments.